# Food-101: v3 - 15 Clases (Desde Cero)

## Objetivos
- Evaluar capacidad del modelo **desde cero** con 15 clases (vs 10 en v2.1)
- Arquitectura más profunda (5 bloques conv) para mayor capacidad
- Augmentation agresivo para manejar variabilidad visual
- Análisis detallado de confusión entre clases

## Mejoras vs v2.1
- **Arquitectura:** 5 bloques conv (32→64→128→256→512) + clasificador de 2 capas
- **Augmentation:** Agresivo (Flip + Rotation + Zoom + Contrast + Brightness)
- **Regularización:** Dropout progresivo (0.25→0.3→0.35→0.4) + L2 aumentado
- **Training:** LR conservador (0.0003), 100 epochs, batch_size 32
- **Análisis:** Top-5 confusiones, accuracy por clase individual

In [ ]:
# Imports
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

## Configuración

In [ ]:
# Hiperparámetros optimizados para 15 clases
IMG_SIZE = 224
BATCH_SIZE = 32  # Reducido de 64 para gradientes más ruidosos
NUM_CLASSES_TO_USE = 15
AUTOTUNE = tf.data.AUTOTUNE

## Cargar Dataset y Verificar Balance

In [ ]:
# Cargar dataset
(train_ds, val_ds), info = tfds.load(
    'food101',
    split=['train', 'validation'],
    with_info=True,
    as_supervised=True
)

class_names = info.features['label'].names
print(f'Dataset total: {len(class_names)} clases')

# Filtrar a 15 clases
train_ds = train_ds.filter(lambda img, label: label < NUM_CLASSES_TO_USE)
val_ds = val_ds.filter(lambda img, label: label < NUM_CLASSES_TO_USE)
num_classes = NUM_CLASSES_TO_USE
class_names = class_names[:NUM_CLASSES_TO_USE]

print(f'\nUsando {num_classes} clases:')
for i, name in enumerate(class_names):
    print(f'  {i}: {name}')

In [ ]:
# Verificar balance del dataset
train_counts = {i: 0 for i in range(NUM_CLASSES_TO_USE)}
val_counts = {i: 0 for i in range(NUM_CLASSES_TO_USE)}

for _, label in train_ds:
    train_counts[int(label)] += 1
for _, label in val_ds:
    val_counts[int(label)] += 1

print(f'\nBalance del Dataset:')
print(f'Training samples: {sum(train_counts.values())}')
print(f'Validation samples: {sum(val_counts.values())}')
print(f'\nMuestras por clase (Train):'); 
for i in range(NUM_CLASSES_TO_USE):
    print(f'  {class_names[i]:25s}: {train_counts[i]:4d} training, {val_counts[i]:4d} validation')

## Preprocessing y Augmentation Agresivo

In [ ]:
# Preprocessing
def preprocess(image, label):
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = image / 255.0
    return image, label

# Augmentation AGRESIVO para 15 clases
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.15),  # ±15 grados
    tf.keras.layers.RandomZoom(0.15),       # ±15% zoom
    tf.keras.layers.RandomContrast(0.2),    # ±20% contraste
    tf.keras.layers.RandomBrightness(0.2),  # ±20% brillo
])

def augment(image, label):
    image = data_augmentation(image, training=True)
    return image, label

In [ ]:
# Mixup mejorado
def mixup(image, label, alpha=0.2):
    batch_size = tf.shape(image)[0]
    dist = tf.compat.v1.distributions.Beta(alpha, alpha)
    lam = dist.sample(1)[0]
    
    indices = tf.random.shuffle(tf.range(batch_size))
    mixed_image = lam * image + (1 - lam) * tf.gather(image, indices)
    
    label_a = tf.one_hot(label, NUM_CLASSES_TO_USE)
    label_b = tf.one_hot(tf.gather(label, indices), NUM_CLASSES_TO_USE)
    mixed_label = lam * label_a + (1 - lam) * label_b
    
    return mixed_image, mixed_label

# CutMix mejorado
def cutmix(image, label, alpha=1.0):
    batch_size = tf.shape(image)[0]
    image_height, image_width = IMG_SIZE, IMG_SIZE
    
    dist = tf.compat.v1.distributions.Beta(alpha, alpha)
    lam = dist.sample(1)[0]
    
    cut_ratio = tf.sqrt(1.0 - lam)
    cut_h = tf.cast(image_height * cut_ratio, tf.int32)
    cut_w = tf.cast(image_width * cut_ratio, tf.int32)
    
    cy = tf.random.uniform([], 0, image_height, dtype=tf.int32)
    cx = tf.random.uniform([], 0, image_width, dtype=tf.int32)
    
    y1 = tf.clip_by_value(cy - cut_h // 2, 0, image_height)
    y2 = tf.clip_by_value(cy + cut_h // 2, 0, image_height)
    x1 = tf.clip_by_value(cx - cut_w // 2, 0, image_width)
    x2 = tf.clip_by_value(cx + cut_w // 2, 0, image_width)
    
    indices = tf.random.shuffle(tf.range(batch_size))
    shuffled_image = tf.gather(image, indices)
    
    mask = tf.concat([
        tf.zeros([batch_size, y1, image_width, 3]),
        tf.concat([
            tf.zeros([batch_size, y2-y1, x1, 3]),
            tf.ones([batch_size, y2-y1, x2-x1, 3]),
            tf.zeros([batch_size, y2-y1, image_width-x2, 3])
        ], axis=2),
        tf.zeros([batch_size, image_height-y2, image_width, 3])
    ], axis=1)
    
    mixed_image = image * (1 - mask) + shuffled_image * mask
    
    area = tf.cast((x2 - x1) * (y2 - y1), tf.float32)
    total_area = tf.cast(image_height * image_width, tf.float32)
    lam_adjusted = 1.0 - (area / total_area)
    
    label_a = tf.one_hot(label, NUM_CLASSES_TO_USE)
    label_b = tf.one_hot(tf.gather(label, indices), NUM_CLASSES_TO_USE)
    mixed_label = lam_adjusted * label_a + (1 - lam_adjusted) * label_b
    
    return mixed_image, mixed_label


In [ ]:
# Pipeline de datos con augmentation agresivo
train_dataset = (
    train_ds
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .cache()
    .shuffle(2000)  # Aumentado de 1000
    .map(augment, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

def apply_mixup_cutmix(images, labels):
    images = tf.cast(images, tf.float32)
    choice = tf.random.uniform([], 0, 1)
    # Mixup: 30%, CutMix: 30%, Normal: 40%
    if choice < 0.30:
        return mixup(images, labels, alpha=0.2)
    elif choice < 0.60:
        return cutmix(images, labels, alpha=1.0)
    else:
        return images, tf.one_hot(labels, NUM_CLASSES_TO_USE)

train_dataset = train_dataset.map(apply_mixup_cutmix, num_parallel_calls=AUTOTUNE)

val_dataset = (
    val_ds
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .cache()
    .batch(BATCH_SIZE)
    .map(lambda x, y: (x, tf.one_hot(y, NUM_CLASSES_TO_USE)))
    .prefetch(AUTOTUNE)
)

## Arquitectura Profunda (5 Bloques Convolucionales)

In [ ]:
# Arquitectura PROFUNDA con regularización progresiva
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),

    # Bloque 1: 32 filtros - SIN dropout (aprende features básicas)
    tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same'),
    tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(2),

    # Bloque 2: 64 filtros - SIN dropout
    tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same'),
    tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(2),

    # Bloque 3: 128 filtros - Dropout 0.25
    tf.keras.layers.Conv2D(128, 3, activation='relu', padding='same'),
    tf.keras.layers.Conv2D(128, 3, activation='relu', padding='same'),
    tf.keras.layers.Conv2D(128, 3, activation='relu', padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(2),
    tf.keras.layers.Dropout(0.25),

    # Bloque 4: 256 filtros - Dropout 0.3
    tf.keras.layers.Conv2D(256, 3, activation='relu', padding='same'),
    tf.keras.layers.Conv2D(256, 3, activation='relu', padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(2),
    tf.keras.layers.Dropout(0.3),

    # Bloque 5: 512 filtros (NUEVO) - Dropout 0.35
    tf.keras.layers.Conv2D(512, 3, activation='relu', padding='same'),
    tf.keras.layers.Conv2D(512, 3, activation='relu', padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(2),
    tf.keras.layers.Dropout(0.35),

    # Clasificador robusto (2 capas Dense)
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(512, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.0003)),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.0003)),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

model.summary()
total_params = model.count_params()
print(f'\nParams: {total_params:,} | Arquitectura: 5 bloques conv + 2-layer classifier')

## Compilación y Learning Rate Scheduling

In [ ]:
# Compilación
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0003),  # Conservador para 15 clases
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=['accuracy']
)

In [ ]:
# LR Scheduler: Warmup + Cosine Annealing
class WarmupCosineDecay(tf.keras.callbacks.Callback):
    def __init__(self, max_lr=0.0003, min_lr=0.000005, warmup_epochs=8, total_epochs=100):
        super().__init__()
        self.max_lr = max_lr
        self.min_lr = min_lr
        self.warmup_epochs = warmup_epochs
        self.total_epochs = total_epochs
        
    def on_epoch_begin(self, epoch, logs=None):
        if epoch < self.warmup_epochs:
            lr = self.max_lr * (epoch + 1) / self.warmup_epochs
        else:
            progress = (epoch - self.warmup_epochs) / (self.total_epochs - self.warmup_epochs)
            lr = self.min_lr + 0.5 * (self.max_lr - self.min_lr) * (1 + tf.cos(np.pi * progress))
        
        optimizer = self.model.optimizer
        if hasattr(optimizer, 'learning_rate'):
            optimizer.learning_rate.assign(lr)
        
        if epoch < 8 or epoch % 10 == 0:
            print(f'Epoch {epoch+1}: LR = {lr:.7f}')

lr_scheduler = WarmupCosineDecay(max_lr=0.0003, min_lr=0.000005, warmup_epochs=8, total_epochs=100)

## Entrenamiento

In [ ]:
# Callbacks
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=25,  # Más paciente para 15 clases
    restore_best_weights=True,
    verbose=1
)

model_checkpoint = tf.keras.callbacks.ModelCheckpoint(
    'best_model_v3_15classes.keras',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=0
)

# ReduceLROnPlateau: Reducir LR si val_loss se estanca
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=8,
    min_lr=0.000001,
    verbose=1
)

EPOCHS = 100

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    callbacks=[early_stopping, lr_scheduler, model_checkpoint, reduce_lr],
    verbose=1
)

## Evaluación Básica

In [ ]:
# Evaluación
test_loss, test_acc = model.evaluate(val_dataset, verbose=0)
print(f'\nValidation Loss: {test_loss:.4f} | Validation Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)')
print(f'Training Accuracy: {history.history["accuracy"][-1]:.4f}')
print(f'Overfitting gap: {(history.history["accuracy"][-1] - test_acc)*100:.2f}%')

In [ ]:
# Gráficas de entrenamiento
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['accuracy'], label='Train', linewidth=2)
ax1.plot(history.history['val_accuracy'], label='Validation', linewidth=2)
ax1.set_title('Accuracy durante el entrenamiento', fontsize=12)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history.history['loss'], label='Train', linewidth=2)
ax2.plot(history.history['val_loss'], label='Validation', linewidth=2)
ax2.set_title('Loss durante el entrenamiento', fontsize=12)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Análisis Avanzado: Confusión entre Clases

In [ ]:
# Obtener predicciones en validación
y_true = []
y_pred = []
y_pred_proba = []

for images, labels_onehot in val_dataset:
    predictions = model.predict(images, verbose=0)
    y_true.extend(tf.argmax(labels_onehot, axis=1).numpy())
    y_pred.extend(np.argmax(predictions, axis=1))
    y_pred_proba.extend(predictions)

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_pred_proba = np.array(y_pred_proba)

print(f'Total predicciones: {len(y_true)}')
print(f'Accuracy: {np.mean(y_true == y_pred):.4f}')

In [ ]:
# Matriz de confusión
cm = confusion_matrix(y_true, y_pred)

# Visualizar matriz de confusión (normalizada)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(cm_norm, interpolation='nearest', cmap=plt.cm.Blues)
ax.figure.colorbar(im, ax=ax)

ax.set(xticks=np.arange(cm_norm.shape[1]),
        yticks=np.arange(cm_norm.shape[0]),
        xticklabels=class_names,
        yticklabels=class_names,
        ylabel='True label',
        xlabel='Predicted label')

plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
ax.set_title('Matriz de Confusión Normalizada (Filas = Ground Truth)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Accuracy por clase
class_accuracy = cm.diagonal() / cm.sum(axis=1)
class_accuracy_df = pd.DataFrame({
    'Class': class_names,
    'Accuracy': class_accuracy,
    'Correct': cm.diagonal(),
    'Total': cm.sum(axis=1)
}).sort_values('Accuracy', ascending=False)

print('\nAccuracy por Clase:')
print(class_accuracy_df.to_string(index=False))
print(f'\nMejor: {class_accuracy_df.iloc[0]["Class"]} ({class_accuracy_df.iloc[0]["Accuracy"]:.2%})')
print(f'Peor: {class_accuracy_df.iloc[-1]["Class"]} ({class_accuracy_df.iloc[-1]["Accuracy"]:.2%})')

In [ ]:
# Encontrar Top-5 pares de clases más confundidas
confusion_pairs = []

for i in range(num_classes):
    for j in range(num_classes):
        if i != j and cm[i, j] > 0:
            confusion_pairs.append({
                'True': class_names[i],
                'Predicted': class_names[j],
                'Count': cm[i, j],
                'Percentage': f'{100 * cm[i, j] / cm[i].sum():.1f}%'
            })

confusion_df = pd.DataFrame(confusion_pairs).sort_values('Count', ascending=False)

print('\n=== TOP-10 CONFUSIONES ENTRE CLASES ===')
print(confusion_df.head(10).to_string(index=False))

In [ ]:
# Classification report
print('\n=== CLASSIFICATION REPORT ===')
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

## Análisis por Confianza

In [ ]:
# Analizar confianza de las predicciones
max_proba = np.max(y_pred_proba, axis=1)
correct_mask = (y_pred == y_true)

print('\n=== ANÁLISIS DE CONFIANZA ===')
print(f'Confianza promedio (todas): {max_proba.mean():.4f}')
print(f'Confianza promedio (correctas): {max_proba[correct_mask].mean():.4f}')
print(f'Confianza promedio (incorrectas): {max_proba[~correct_mask].mean():.4f}')

# Predicciones con alta confianza pero incorrectas
high_confidence_wrong = np.where((max_proba > 0.9) & (~correct_mask))[0]
print(f'\nPredicciones con >90% confianza INCORRECTAS: {len(high_confidence_wrong)}')

if len(high_confidence_wrong) > 0:
    print('\nEjemplos (primeros 5):')
    for idx in high_confidence_wrong[:5]:
        print(f'  True: {class_names[y_true[idx]]}, Predicted: {class_names[y_pred[idx]]} (conf: {max_proba[idx]:.3f})')

## Conclusiones

In [ ]:
print(f'\n{"="*70}')
print(f'=== RESULTADOS FINALES: v3 CON 15 CLASES ===')
print(f'{"="*70}')
print(f'\nAccuracy de validación: {test_acc*100:.2f}%')
print(f'Accuracy de entrenamiento: {history.history["accuracy"][-1]*100:.2f}%')
print(f'Overfitting gap: {(history.history["accuracy"][-1] - test_acc)*100:.2f} puntos')
print(f'\nEpochs completados: {len(history.history["loss"])}/{EPOCHS}')
print(f'Parámetros del modelo: {total_params:,}')
print(f'\nClase con mejor accuracy: {class_accuracy_df.iloc[0]["Class"]} ({class_accuracy_df.iloc[0]["Accuracy"]:.2%})')
print(f'Clase con peor accuracy: {class_accuracy_df.iloc[-1]["Class"]} ({class_accuracy_df.iloc[-1]["Accuracy"]:.2%})')
print(f'\nTop confusión: {confusion_df.iloc[0]["True"]} → {confusion_df.iloc[0]["Predicted"]} ({confusion_df.iloc[0]["Count"]} veces)')
print(f'\nModelo guardado: best_model_v3_15classes.keras')
print(f'{"="*70}')